# 📬 Notebook 4: Queues and Load Shedding

When traffic spikes exceed capacity, use queues to absorb bursts and load shedding to gracefully degrade.

## Learning Objectives

By the end of this notebook, you'll understand:
- Using queues to absorb write bursts
- Async write patterns
- Load shedding strategies
- Real-world examples (Uber, Strava)

In [ ]:
import redis
import json
import time
import random
from datetime import datetime
from typing import Optional, Callable

# Seeded so the numbers printed below are the same on every run.
random.seed(42)

r = redis.Redis(host='localhost', port=6379, decode_responses=True)
r.flushall()

print("✅ Connected to Redis!")
print("📊 Open RedisInsight: http://localhost:5540")

## 📬 Write Queues

In [ ]:
print("📬 Queue-Based Write Pattern")
print("=" * 60)
print("""
PROBLEM: Bursty traffic overwhelms database
─────────────────────────────────────────────────────────────
              BURST!                    
    Writers ━━━━━━━━━━━━━━━━━━━━━━> [Database] 💥 Overloaded!
              10,000 writes/sec          Max: 1,000/sec

─────────────────────────────────────────────────────────────

SOLUTION: Queue absorbs bursts, workers drain at safe rate
─────────────────────────────────────────────────────────────
              BURST!           Steady drain
    Writers ━━━━━━━> [Queue] ━━━━━━━━━━━━━> [Database] ✓
              10K/sec   Buffer    1K/sec      Happy!

• Queue grows during bursts
• Queue drains during calm periods
• Database sees steady write rate
""")

In [ ]:
class WriteQueue:
    def __init__(self, queue_name: str, max_size: int = 10000):
        self.queue_name = queue_name
        self.max_size = max_size
        self.stats = {"enqueued": 0, "dropped": 0, "processed": 0}
    
    def enqueue(self, data: dict) -> bool:
        current_size = r.llen(self.queue_name)
        if current_size >= self.max_size:
            self.stats["dropped"] += 1
            return False
        
        data["queued_at"] = datetime.now().isoformat()
        r.rpush(self.queue_name, json.dumps(data))
        self.stats["enqueued"] += 1
        return True
    
    def dequeue(self, batch_size: int = 1) -> list:
        items = []
        for _ in range(batch_size):
            item = r.lpop(self.queue_name)
            if item:
                items.append(json.loads(item))
                self.stats["processed"] += 1
            else:
                break
        return items
    
    def size(self) -> int:
        return r.llen(self.queue_name)

print("📊 Simulating Bursty Traffic")
print("=" * 60)

queue = WriteQueue("location_updates", max_size=1000)

# A real burst is bigger than the queue can hold, which is exactly
# when load shedding kicks in. Push 1,500 writes into a 1,000-slot
# queue so you can watch the "dropped" counter climb.
print("\n🌊 Simulating burst: 1,500 writes into a queue capped at 1,000")
for i in range(1500):
    queue.enqueue({"user_id": i, "lat": 40.7 + random.random(), "lng": -74.0 + random.random()})

print(f"   Queue size: {queue.size()}")
print(f"   Enqueued: {queue.stats['enqueued']}")
print(f"   Dropped: {queue.stats['dropped']}")

print("\n⏱️ Simulating steady processing...")
while queue.size() > 0:
    batch = queue.dequeue(batch_size=50)
    time.sleep(0.01)

print(f"   Processed: {queue.stats['processed']}")
print("\n✅ Database saw steady 50 writes/batch instead of a 1,500-write spike!")
print("   The dropped writes are the *cost* of protecting the database:")
print("   you trade completeness for availability.")

# The whole section only teaches something if the queue actually overflows.
# 1,500 writes into 1,000 slots must accept 1,000 and shed exactly 500.
assert queue.stats["enqueued"] == 1000, (
    f"a 1,000-slot queue must accept exactly 1,000 writes, got "
    f"{queue.stats['enqueued']}"
)
assert queue.stats["dropped"] == 500, (
    f"expected 500 shed writes, got {queue.stats['dropped']} -- if this is 0 "
    f"the burst is no longer bigger than the queue and load shedding never fires"
)
assert queue.stats["processed"] == 1000, (
    f"everything accepted must eventually drain, got "
    f"{queue.stats['processed']} of {queue.stats['enqueued']}"
)


## 🚮 Load Shedding

In [ ]:
print("🚮 Load Shedding Strategies")
print("=" * 60)
print("""
When queue is full, strategically DROP less important writes.

STRATEGIES:
─────────────────────────────────────────────────────────────

1. DROP OLDEST (Uber location updates)
   ┌────────────────────────────────┐
   │ Old │ Old │ Old │ New │ New │  │ ← New
   └──────▲─────────────────────────┘
          └── Drop stale locations

2. DROP NEWEST (Preserve order)
   ┌────────────────────────────────┐
   │ 1st │ 2nd │ 3rd │ 4th │ 5th │ │ ← Reject new
   └────────────────────────────────┘
       First come, first served

3. DROP BY PRIORITY
   ┌──────────────────────────────────────┐
   │ P1! │ P1! │ P2  │ P3  │ P3  │ P3 │  │
   └─────────────────────▲────────────────┘
                         └── Drop low priority first

4. SAMPLE (Strava segments)
   Keep every Nth location point:
   ✓ • • ✓ • • ✓ • • ✓ • • 
""")

In [ ]:
class PriorityQueue:
    def __init__(self):
        self.queue_name = "priority_writes"
        self.stats = {"high": 0, "medium": 0, "low": 0, "dropped": 0}
    
    def enqueue(self, data: dict, priority: str = "medium") -> bool:
        score = {"high": 3, "medium": 2, "low": 1}[priority]
        r.zadd(self.queue_name, {json.dumps(data): score})
        self.stats[priority] += 1
        return True
    
    def shed_load(self, keep_count: int) -> dict:
        """Drop lowest-scored members until only keep_count remain.

        Returns a {priority: count} breakdown of what was actually dropped.
        That breakdown matters: shedding walks UP the score order, so once it
        runs out of low-priority items it starts eating medium ones. Reporting
        a flat "dropped N low-priority items" would be a lie the moment the
        overload is bigger than your low-priority backlog.
        """
        total = r.zcard(self.queue_name)
        if total <= keep_count:
            return {}
        # zpopmin returns (member, score) pairs, lowest score first.
        popped = r.zpopmin(self.queue_name, total - keep_count)
        self.stats["dropped"] += len(popped)
        return _by_priority(popped)


NAME_FOR_SCORE = {3: "high", 2: "medium", 1: "low"}


def _by_priority(members) -> dict:
    """Count (member, score) pairs by priority name, omitting empty buckets."""
    counts = {"high": 0, "medium": 0, "low": 0}
    for _member, score in members:
        counts[NAME_FOR_SCORE[int(score)]] += 1
    return {k: v for k, v in counts.items() if v}


print("🎯 Priority-Based Load Shedding")
print("=" * 60)

pq = PriorityQueue()
r.delete(pq.queue_name)

for i in range(30):
    pq.enqueue({"id": i, "type": "low_priority"}, "low")
for i in range(20):
    pq.enqueue({"id": i, "type": "medium_priority"}, "medium")
for i in range(10):
    pq.enqueue({"id": i, "type": "high_priority"}, "high")

print(f"\n📊 Before shedding:")
print(f"   High priority: {pq.stats['high']}")
print(f"   Medium priority: {pq.stats['medium']}")
print(f"   Low priority: {pq.stats['low']}")
print(f"   Total in queue: {r.zcard(pq.queue_name)}")

dropped = pq.shed_load(keep_count=25)
survivors = _by_priority(r.zrange(pq.queue_name, 0, -1, withscores=True))

print(f"\n🚮 After shedding down to 25 items:")
print(f"   Dropped {sum(dropped.values())}: {dropped}")
print(f"   Survived {sum(survivors.values())}: {survivors}")

print("\n✅ Every high-priority write survived.")
print("⚠️  But read the breakdown again: shedding 35 of 60 items ate all 30")
print("   low-priority writes AND 5 medium ones. Priority shedding does not")
print("   mean 'only unimportant things get dropped' -- it means 'the least")
print("   important thing still in the queue gets dropped first'. If the")
print("   overload is deep enough, it will reach your important traffic, and")
print("   the only honest thing to do is alert on which tier you are eating.")

# The guarantee this section sells is "high priority survives". Pin it, and
# pin the exact composition so the printed numbers can never drift from truth.
assert survivors.get("high", 0) == 10, (
    f"all 10 high-priority writes must survive shedding, got {survivors}"
)
assert survivors.get("low", 0) == 0, (
    f"low priority must be shed before anything else, got {survivors}"
)
assert dropped == {"low": 30, "medium": 5}, (
    f"expected to shed all 30 low + 5 medium, got {dropped}"
)


## 🚗 Real World: Uber Location Updates

In [ ]:
print("🚗 Uber-Style Location Updates")
print("=" * 60)
print("""
Uber receives millions of GPS updates per second.
Not all updates are equally important!

Strategy: Only keep LATEST location per driver
─────────────────────────────────────────────────────────────

Driver 123 sends:
  10:00:01 → Lat: 40.7128  ← Overwritten
  10:00:02 → Lat: 40.7130  ← Overwritten  
  10:00:03 → Lat: 40.7132  ← KEPT (latest)

Result: 3 updates → 1 write to database!
""")

In [ ]:
class LatestOnlyQueue:
    def __init__(self, prefix: str = "driver_loc"):
        self.prefix = prefix
        self.updates_received = 0
        self.unique_keys = set()
    
    def update_location(self, driver_id: int, lat: float, lng: float):
        key = f"{self.prefix}:{driver_id}"
        r.hset(key, mapping={
            "lat": lat,
            "lng": lng,
            "updated_at": datetime.now().isoformat()
        })
        self.updates_received += 1
        self.unique_keys.add(key)
    
    def get_stats(self) -> dict:
        return {
            "updates_received": self.updates_received,
            "unique_drivers": len(self.unique_keys),
            "write_reduction": f"{(1 - len(self.unique_keys)/self.updates_received)*100:.1f}%"
        }

print("🚗 Simulating Driver Location Updates")
print("=" * 60)

location_queue = LatestOnlyQueue()

print("\n📍 Each of 100 drivers sends 50 location updates...")
for driver_id in range(100):
    base_lat, base_lng = 40.7 + random.random() * 0.1, -74.0 + random.random() * 0.1
    for update in range(50):
        lat = base_lat + random.random() * 0.001
        lng = base_lng + random.random() * 0.001
        location_queue.update_location(driver_id, lat, lng)

stats = location_queue.get_stats()
print(f"\n📊 Results:")
print(f"   Updates received: {stats['updates_received']}")
print(f"   Unique drivers: {stats['unique_drivers']}")
print(f"   Write reduction: {stats['write_reduction']}")

reduction = 1 - stats["unique_drivers"] / stats["updates_received"]

print(f"\n✅ 5,000 GPS pings collapse to {stats['unique_drivers']} rows to flush "
      f"-- a {stats['updates_received'] // stats['unique_drivers']}x write reduction.")
print("   Be precise about what was reduced: the 5,000 HSETs still hit Redis.")
print("   What PostgreSQL never sees is the 4,900 superseded positions -- it")
print("   only ever writes the current one per driver.")
print("   This works because location is *last-writer-wins* state, not an")
print("   event log. Never do this to data where every entry matters.")

assert stats["updates_received"] == 5000, stats
assert stats["unique_drivers"] == 100, (
    f"100 drivers must collapse to 100 keys, got {stats}"
)
assert reduction > 0.95, f"expected >95% write reduction, got {reduction:.1%}"


## ☠️ Dead-Letter Queues & Backpressure

Real queue-based systems need two more ideas beyond "enqueue + drain":

### Dead-Letter Queue (DLQ)

If a worker fails to process an item **N times in a row** (e.g., a malformed
payload, a bug, a downstream outage), park it in a separate **dead-letter
queue** instead of retrying forever. A human or a repair job can inspect
and replay those items later.

```
main queue ---> worker --X fail 3x ---> dead-letter queue
                                        (triage later)
```

Without a DLQ, one poison message can block the whole pipeline.

### Backpressure

When the queue is growing faster than workers drain it, you're heading
for trouble. Healthy systems push that pressure **back to the producer**:

- **Reject with `429 Too Many Requests`** — the client retries with backoff.
- **Slow down the producer** — e.g., an SDK throttles its own calls.
- **Shed load** (previous section) — drop less important writes.

The golden rule: **an unbounded queue is a bug**, not a feature. It just
hides the overload until memory runs out.


In [ ]:
print("☠️ Simulating a dead-letter queue")
print("=" * 60)

r.delete("work_queue", "dlq")
MAX_RETRIES = 3

def enqueue_work(item: dict, retries: int = 0):
    item["retries"] = retries
    r.rpush("work_queue", json.dumps(item))

# Producer: 10 items, item #4 is poisoned (always fails)
for i in range(10):
    enqueue_work({"id": i, "poisoned": i == 4})

def process(item: dict):
    if item.get("poisoned"):
        raise RuntimeError("cannot parse payload")

processed, dead = 0, 0
while r.llen("work_queue") > 0:
    raw = r.lpop("work_queue")
    item = json.loads(raw)
    try:
        process(item)
        processed += 1
    except Exception as e:
        if item["retries"] + 1 >= MAX_RETRIES:
            r.rpush("dlq", json.dumps({"item": item, "error": str(e)}))
            dead += 1
        else:
            enqueue_work(item, retries=item["retries"] + 1)

print(f"\n✅ Processed: {processed}")
print(f"☠️ Sent to DLQ: {dead}")
print(f"   DLQ contents: {r.lrange('dlq', 0, -1)}")

# The poison message must be isolated, not silently dropped and not blocking
# the other 9. If either of those changes, the DLQ is not doing its job.
assert processed == 9, f"the 9 healthy items must all succeed, got {processed}"
assert dead == 1, f"exactly the poison item belongs in the DLQ, got {dead}"
assert r.llen("dlq") == 1, f"DLQ should hold 1 item, holds {r.llen('dlq')}"
parked = json.loads(r.lindex("dlq", 0))
assert parked["item"]["id"] == 4, f"wrong item parked: {parked}"
assert parked["item"]["retries"] == MAX_RETRIES - 1, (
    f"item should have been retried {MAX_RETRIES} times before parking, "
    f"retries field says {parked['item']['retries']}"
)
print(f"   Item 4 was attempted {MAX_RETRIES} times, then parked instead of "
      f"blocking the other 9.")


## 💥 What a Write-Behind Queue Actually Loses When a Worker Crashes

Every demo so far dequeued with `LPOP`: take the item off the queue, *then* do
the work. That is **at-most-once** delivery, and the gap between those two
steps is a hole you can drive a truck through:

```
LPOP  ─────────────▶  item now exists only in the worker's RAM
       💥 process dies
                      gone from Redis, never written to the database
```

Nothing errors. Nothing is left over to retry. There is no queue depth alarm,
because the item left the queue exactly as designed. The write simply never
happened, and the only way you find out is a customer.

The fix is to make the dequeue **non-destructive**: atomically move the item to
a per-worker *in-flight* list (`LMOVE`), do the work, and only then `LREM` it.
A crash leaves the item sitting in the in-flight list where a recovery pass can
push it back. That is **at-least-once**, which shifts the burden onto your
handler: it can now be called twice for the same item, so it must be
idempotent (unique key, upsert, dedupe table).

There is no third option. A queue and a database that do not share a
transaction can give you at-most-once or at-least-once — never exactly-once.
"Exactly-once" in a product datasheet always means at-least-once delivery plus
an idempotent consumer.

In [ ]:
print("💥 Crash mid-processing: LPOP (at-most-once) vs LMOVE (at-least-once)")
print("=" * 60)

CRASH_ON = 2   # the worker dies while holding this item


def load(queue: str, n: int = 5):
    r.delete(queue, f"{queue}:inflight")
    for i in range(n):
        r.rpush(queue, json.dumps({"id": i}))


# --- 1. LPOP: pop first, write second ------------------------------------
load("amo_queue")
written_amo = []
for _ in range(5):
    raw = r.lpop("amo_queue")
    item = json.loads(raw)
    if item["id"] == CRASH_ON:
        break                      # 💥 dies here: after the pop, before the write
    written_amo.append(item["id"])

still_queued = [json.loads(x)["id"] for x in r.lrange("amo_queue", 0, -1)]
lost = 5 - len(written_amo) - len(still_queued)

print("\n1️⃣ LPOP, worker crashes while holding item 2:")
print(f"   written to the DB : {written_amo}")
print(f"   still in the queue: {still_queued}")
print(f"   in flight anywhere: nothing -- item {CRASH_ON} only existed in the dead process")
print(f"   ❌ silently lost  : {lost} item(s), with no error and no leftover")

# --- 2. LMOVE + LREM: reserve first, delete after the write ---------------
load("alo_queue")
written_alo = []
for _ in range(5):
    # Atomic: the item is never in zero places. It is in the queue, or in the
    # in-flight list, or (after LREM) durably written.
    raw = r.lmove("alo_queue", "alo_queue:inflight", "LEFT", "RIGHT")
    if raw is None:
        break
    item = json.loads(raw)
    if item["id"] == CRASH_ON:
        break                      # 💥 same crash, same instant
    written_alo.append(item["id"])
    r.lrem("alo_queue:inflight", 1, raw)

print("\n2️⃣ LMOVE + LREM, same crash:")
print(f"   written to the DB : {written_alo}")
print(f"   still in the queue: {[json.loads(x)['id'] for x in r.lrange('alo_queue', 0, -1)]}")
print(f"   in flight         : {[json.loads(x)['id'] for x in r.lrange('alo_queue:inflight', 0, -1)]}")

# Recovery: a supervisor (or the worker on restart) requeues anything that has
# been in flight longer than the processing timeout.
recovered = 0
while r.llen("alo_queue:inflight") > 0:
    r.lmove("alo_queue:inflight", "alo_queue", "RIGHT", "LEFT")
    recovered += 1

survived = len(written_alo) + r.llen("alo_queue")
print(f"   🔁 recovery requeued {recovered} in-flight item(s)")
print(f"   queue after recovery: {[json.loads(x)['id'] for x in r.lrange('alo_queue', 0, -1)]}")
print(f"   ✅ accounted for   : {survived}/5")

# Reproduce the failure, then show the fix. Both halves have to keep holding.
assert lost == 1, (
    f"the LPOP path must lose exactly the in-flight item -- if it lost {lost}, "
    f"this cell is no longer demonstrating at-most-once data loss"
)
assert survived == 5, (
    f"LMOVE + recovery must lose nothing, only accounted for {survived}/5"
)

print("""
💡 The bill for at-least-once: if the crash had landed *after* the DB write but
   *before* the LREM, item 2 would be processed twice on recovery. You cannot
   close that window with a queue alone -- you close it in the handler, with a
   unique constraint or an upsert keyed on the item id.

   And note what the in-flight list needs to be useful in production: a
   visibility timeout (how long before a held item is presumed dead) and a
   redelivery counter, so a message that crashes its worker every time ends up
   in the DLQ from the previous section instead of looping forever.
""")


## 🧪 Quick Quiz

1. **When should you drop oldest vs newest items?**

2. **Why does Uber only keep the latest location per driver?**

3. **What's the trade-off of queue-based writes?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Drop oldest vs newest:")
print("   Drop OLDEST: Real-time data (locations, metrics)")
print("   Drop NEWEST: Order matters (transactions, logs)")
print()
print("2. Uber keeps latest only because:")
print("   - Old locations are stale/useless")
print("   - Only current position matters for matching")
print("   - Reduces writes by 50x or more")
print()
print("3. Queue trade-offs:")
print("   Pros: Absorbs bursts, steady DB load")
print("   Cons: Eventual consistency (delay)")
print("         Queue failure = data loss")

## 📚 Summary

### Key Takeaways

1. **Queues absorb bursts** - Buffer writes during spikes
2. **Load shedding** - Strategically drop less important writes
3. **Latest-only** - For real-time data, overwrite instead of append
4. **Priority queues** - Preserve important writes under load
5. **Trade-off** - Eventual consistency for better availability

### Next Up

In **Notebook 5**, we'll learn about batching and aggregation:
- Combining multiple writes into one
- Hierarchical counters
- Like/view aggregation patterns